# 03 · Reproducible regression tables

**Module:** 4 · **Time:** 45 minutes · **Prerequisites:** 02 Standard errors and tests

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/modules/04_econometrics/03_regression_tables.ipynb) · [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ningzhanglab/Python-for-Beginners/main?filepath=modules%2F04_econometrics%2F03_regression_tables.ipynb)

**In Colab choose File → Save a copy in Drive before editing.** Badges target `main` after merge; this is a private repository and draft PR.

## Learning goals

- **B**uild a table from live model result objects.
- **C**learly label coefficient estimates and standard-error choices.
- **E**xport regression results to CSV and LaTeX without retyping estimates.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

rng = np.random.default_rng(512)
n_obs = 150
cluster_id = np.repeat(np.arange(30), 5)
schooling = rng.integers(10, 19, size=n_obs)
experience = rng.integers(0, 25, size=n_obs)
cluster_shocks = rng.normal(0, 2, size=30)
person_shocks = rng.normal(0, 3, size=n_obs)

In [ ]:
wage_index = (5 + 1.6 * schooling + 0.5 * experience
              + cluster_shocks[cluster_id] + person_shocks)
survey = pd.DataFrame({"wage_index": wage_index,
                       "schooling": schooling, "experience": experience,
                       "cluster_id": cluster_id})
print("Fictional workers:", len(survey))
print(survey.head(3))

## Why this matters

A research paper should not contain manually copied coefficient values that can drift from the analysis. Model result objects are the source of truth. A descriptive table should identify outcome, regressor units, sample size, and how standard errors were calculated.

## Estimate two specifications

Compare a model with schooling alone with one adjusting for experience. Both use HC3 uncertainty and the **same simulated 150-person dataset**; this comparison does not establish a causal effect.

In [ ]:
model_1 = smf.ols("wage_index ~ schooling",
                  data=survey).fit(cov_type="HC3")
model_2 = smf.ols("wage_index ~ schooling + experience",
                  data=survey).fit(cov_type="HC3")
print("Sample sizes:", int(model_1.nobs), int(model_2.nobs))

## Create a transparent coefficient/SE table

Align parameter names explicitly. `NaN` for experience in Model 1 means it was not included, **not** an estimate of zero.

In [ ]:
terms = ["Intercept", "schooling", "experience"]
regression_table = pd.DataFrame({
    "M1_coef": model_1.params.reindex(terms),
    "M1_HC3_SE": model_1.bse.reindex(terms),
    "M2_coef": model_2.params.reindex(terms),
    "M2_HC3_SE": model_2.bse.reindex(terms),
})
print(regression_table.round(3))

## Report model metadata

Record the outcome and inference method alongside statistics. Here R-squared and N are diagnostics rather than rows of coefficient values.

In [ ]:
metadata = pd.DataFrame({
    "model": ["M1", "M2"],
    "outcome": ["wage_index", "wage_index"],
    "N": [int(model_1.nobs), int(model_2.nobs)],
    "R2": [model_1.rsquared, model_2.rsquared],
    "SE_method": ["HC3", "HC3"],
})
print(metadata.to_string(index=False))

## Export to research-friendly formats

CSV is easy to inspect and reuse. LaTeX can feed into manuscript workflows; annotate your table caption with units, estimator and covariates.

In [ ]:
csv_text = regression_table.to_csv(index=True)
latex_text = regression_table.to_latex(float_format="%.3f")
print(csv_text.splitlines()[0])
print("LaTeX table created:", "\\begin{tabular}" in latex_text)

## If you know Stata/R

| Task | Python | Stata | R |
| --- | --- | --- | --- |
| Read coefficients | `result.params` | `matrix list e(b)` | `coef(model)` |
| Read SEs | `result.bse` | `matrix list e(V)` | `sqrt(diag(vcov(model)))` |
| Export tables | `DataFrame.to_latex()` | `estimates table` (display) | `modelsummary` (add-on package) |

Always label the uncertainty estimator and note when covariates change between specifications.

## Try it

Exercises begin with `None` in the learner notebook. Checks offer hints rather than raising errors, while completed solutions must pass every check.

### ★ Exercise 1 — Define the table's row names

Create `answer_terms` as the list `['Intercept', 'schooling', 'experience']`.

In [ ]:
answer_terms = None

In [ ]:
if isinstance(answer_terms, list) and answer_terms == terms:
    print("✓ Regression rows selected")
else:
    print("Hint: list the intercept and both explanatory variables.")

### ★★ Exercise 2 — Extract and align coefficients

Create `m1_coefs` as Model 1 coefficients reindexed to include all three terms.

In [ ]:
m1_coefs = None

In [ ]:
expected_coefs = model_1.params.reindex(terms)
if (isinstance(m1_coefs, pd.Series) and
    m1_coefs.index.equals(expected_coefs.index) and
    np.allclose(m1_coefs.to_numpy(), expected_coefs.to_numpy(), equal_nan=True)):
    print("✓ Model coefficients aligned")
else:
    print("Hint: use model_1.params.reindex(terms) to preserve missing terms.")

### ★★ Exercise 3 — Record model sample size

Set `m2_n` to Model 2's number of observations as an integer.

In [ ]:
m2_n = None

In [ ]:
if isinstance(m2_n, (int, np.integer)) and m2_n == len(survey):
    print("✓ Sample size recorded")
else:
    print("Hint: read the fitted model's .nobs property.")

### ★★★ Exercise 4 — Export the table as CSV text

Set `export_csv` to the CSV representation of the coefficient table, including its row index.

In [ ]:
export_csv = None

In [ ]:
if (isinstance(export_csv, str) and
    "M2_HC3_SE" in export_csv.splitlines()[0] and
    "schooling" in export_csv):
    print("✓ Reproducible table exported")
else:
    print("Hint: call regression_table.to_csv(index=True).")

## Common errors

- Reusing standard errors from a different model produces an internally inconsistent table.
- Do not print an absent variable as zero: missing coefficients should remain blank or missing.
- If sample sizes differ due to missing data, the two model columns do not use identical observations.
- CSV/LaTeX exports require a caption and notes about outcome units, inference and sample definition.

## Recap

- Read tables directly from fitted models, not from copied outputs.
- Use stable term order and explicit uncertainty labels.
- Record N, R-squared, and the standard-error method with each specification.

## Next steps

Continue to fixed-effects methods once that lesson is tested and added. See the [statsmodels results documentation](https://www.statsmodels.org/stable/regression.html).

[Report a content error](https://github.com/ningzhanglab/Python-for-Beginners/issues/new?template=content_error.md) with this lesson path.